# Part 2 - SQL: verify the numbers and find where no-shows cluster
**Project: CareLoop Clinics - No-Show Guard**

Numbers that a manager acts on must be reproducible from the database, and they must agree
with the Python numbers from Part 1. In this notebook we load Part 1's output into a database
called `noshowguard` and answer business questions with SQL.

| Step | What we do | SQL ideas |
|---|---|---|
| 2.1 | Create the tables, load the data, test the primary key | `CREATE TABLE`, `PRIMARY KEY`, `FOREIGN KEY` |
| 2.2 | No-show rate per clinic | `JOIN`, `GROUP BY`, `HAVING` |
| 2.3 | No-show rate by lead-time band | `CASE WHEN` |
| 2.4 | Month-by-month trend | CTE (`WITH`), `LAG()` |
| 2.5 | Rank clinics inside each city | CTE, `RANK() OVER (PARTITION BY ...)`, scalar subquery |

Every query here is also saved in `sql/02_queries.sql`.

## Setup
The notebook talks to a **local MySQL 8 server**. The connection details are read from
environment variables (or from a `.env` file that is *not* uploaded to GitHub), so no password
is ever written in this notebook.

If no MySQL server can be reached (for example on Google Colab), the notebook falls back to
**SQLite**, a small database that lives inside Python. Every query below runs on both.

In [1]:
# Install the libraries this notebook needs (safe to re-run).
%pip install -q --disable-pip-version-check pandas mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sqlite3
from decimal import Decimal
from pathlib import Path

import pandas as pd

# On Google Colab the project files are not there yet, so we download (clone) the repo first.
if "google.colab" in str(get_ipython()) and not Path("MASAI-CAPSTONE-PROJ").exists():
    os.system("git clone -q https://github.com/Advay4706/MASAI-CAPSTONE-PROJ")
if Path("MASAI-CAPSTONE-PROJ").exists():
    os.chdir("MASAI-CAPSTONE-PROJ")

BASE = Path(".") if Path("data").exists() else Path("..")   # the project's top folder
OUT = BASE / "outputs"


def load_env_file(path):
    """Read KEY=VALUE lines from a .env file into environment variables (if the file exists)."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        if "=" in line and not line.strip().startswith("#"):
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())


load_env_file(BASE / ".env")

# Try MySQL first. If anything goes wrong (no password set, no server), use SQLite instead.
try:
    import mysql.connector
    conn = mysql.connector.connect(
        host=os.environ.get("MYSQL_HOST", "127.0.0.1"),
        port=int(os.environ.get("MYSQL_PORT", "3306")),
        user=os.environ.get("MYSQL_USER", "root"),
        password=os.environ["MYSQL_PASSWORD"],     # never typed into the notebook
        connection_timeout=5,
    )
    USING = "MySQL"
    DB_ERRORS = (mysql.connector.Error,)
except Exception as err:                           # print only the error type, never its details
    print(f"MySQL not available ({type(err).__name__}) -> using SQLite instead")
    conn = sqlite3.connect(":memory:")             # a temporary database in memory
    USING = "SQLite"
    DB_ERRORS = (sqlite3.Error,)


def run(sql):
    """Run one statement that returns no rows (CREATE, INSERT, ...)."""
    cursor = conn.cursor()
    cursor.execute(sql)
    conn.commit()
    cursor.close()


def tidy(value):
    """MySQL returns exact Decimal numbers; turn them into plain int/float for display."""
    if isinstance(value, Decimal):
        return int(value) if value.as_tuple().exponent >= 0 else float(value)
    return value


def query(sql):
    """Run a SELECT and return the result as a pandas DataFrame."""
    cursor = conn.cursor()
    cursor.execute(sql)
    columns = [column[0] for column in cursor.description]
    rows = [[tidy(v) for v in row] for row in cursor.fetchall()]
    cursor.close()
    return pd.DataFrame(rows, columns=columns)


print("Database engine in use:", USING)
print("Server version:", query("SELECT VERSION() AS v" if USING == "MySQL" else "SELECT sqlite_version() AS v")["v"][0])

Database engine in use: MySQL
Server version: 8.4.9


## Step 2.1 - Build and load the database

**What:** create the database `noshowguard`, create two tables, run `load_data.sql` from
Part 1 (998 `INSERT` statements), and check the counts. Then try to insert a second row with
an `appointment_id` that already exists and record the error.

**Why:** the table definitions are the database's own quality rules.

- `PRIMARY KEY` on `appointment_id` = every appointment has one unique, non-empty ID. The
  database itself refuses a duplicate - we do not have to remember to check.
- `FOREIGN KEY (clinic_id) REFERENCES clinics(clinic_id)` = an appointment can only point to a
  clinic that exists. That is why clinics are loaded first.
- `NOT NULL` = this column may not be empty.
- Types: `VARCHAR(n)` text up to n characters, `INT` whole number, `DATE` a calendar date,
  `DECIMAL(5,1)` an exact number with 5 digits in total and 1 after the decimal point.

In [3]:
CREATE_CLINICS = """
CREATE TABLE clinics (
  clinic_id VARCHAR(5) PRIMARY KEY, clinic_name VARCHAR(40) NOT NULL, city VARCHAR(20) NOT NULL,
  tier VARCHAR(10), latitude DECIMAL(8,4), longitude DECIMAL(8,4), temp_c DECIMAL(5,1));
"""

CREATE_APPOINTMENTS = """
CREATE TABLE appointments (
  appointment_id VARCHAR(8) PRIMARY KEY, patient_id VARCHAR(8) NOT NULL, clinic_id VARCHAR(5) NOT NULL,
  booked_on DATE, appt_date DATE, age INT, gender VARCHAR(1), distance_km DECIMAL(5,1), fee_inr INT,
  booking_channel VARCHAR(10), visit_type VARCHAR(12), reminder_sent VARCHAR(3), prior_no_shows INT,
  status VARCHAR(10), lead_days INT, no_show INT,
  FOREIGN KEY (clinic_id) REFERENCES clinics(clinic_id));
"""

if USING == "MySQL":
    # Start from an empty database so this notebook can be re-run from the top.
    run("DROP DATABASE IF EXISTS noshowguard")
    run("CREATE DATABASE noshowguard")
    run("USE noshowguard")

run(CREATE_CLINICS)          # parent table first
run(CREATE_APPOINTMENTS)     # child table second (it refers to clinics)
print("Tables created.")

Tables created.


In [4]:
# Run load_data.sql: one INSERT statement per line, clinics first.
statements = [line for line in (OUT / "load_data.sql").read_text(encoding="utf-8").splitlines()
              if line.startswith("INSERT INTO")]

cursor = conn.cursor()
for statement in statements:
    cursor.execute(statement)
conn.commit()                # make the inserts permanent
cursor.close()
print("INSERT statements run:", len(statements))

INSERT statements run: 998


In [5]:
Q_COUNTS = """
SELECT
  (SELECT COUNT(*) FROM clinics)          AS clinics,
  (SELECT COUNT(*) FROM appointments)     AS appointments,
  (SELECT SUM(no_show) FROM appointments) AS no_shows;
"""

counts = query(Q_COUNTS)
display(counts)

,clinics,appointments,no_shows
0,8,990,218


### Test the primary key
`A0001` is already in the table. Inserting another row with the same ID must fail.
We catch the error with `try/except` so the notebook keeps running, and print the message.

In [6]:
Q_DUPLICATE = """
INSERT INTO appointments VALUES
  ('A0001','P999','C01','2025-01-01','2025-01-02',30,'F',5.0,500,'App','Consultation','Yes',0,'Attended',1,0);
"""

try:
    run(Q_DUPLICATE)
    print("The insert worked - that would be a problem!")
except DB_ERRORS as err:
    conn.rollback()                      # undo the failed statement
    print("Insert refused.")
    print("Error type:", type(err).__name__)
    print("Error text:", err)

print("\nRows in appointments after the failed insert:",
      query("SELECT COUNT(*) AS n FROM appointments")["n"][0])

Insert refused.
Error type: IntegrityError
Error text: 1062 (23000): Duplicate entry 'A0001' for key 'appointments.PRIMARY'

Rows in appointments after the failed insert: 990


**Reading the result**

- 8 clinics, 990 appointments, 218 no-shows - exactly the numbers from Part 1 (Step 1.4).
  Two different tools agree, which is what makes the numbers trustworthy.
- The duplicate insert fails with MySQL error **1062, "Duplicate entry 'A0001' for key
  'appointments.PRIMARY'"** and the table still has 990 rows. (SQLite words the same refusal
  as "UNIQUE constraint failed".) The same rule is what protects the table against the
  duplicates we had to remove by hand in Step 1.2.